In [ ]:
# # 1. Check what Colab gave you
# import torch
# TORCH_VER = torch.__version__.split('+')[0] # '2.3.1'
# CUDA_VER = 'cu' + torch.version.cuda.replace('.', '') # 'cu121'
# print(f"Detected: torch={TORCH_VER} {CUDA_VER}")

# # 2. Install PyG extensions with matching wheels
# !pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html

# # 3. Install fairchem-core 2.0.0 if you haven't already
# !pip install git+https://github.com/facebookresearch/fairchem.git@fairchem_core-2.0.0#subdirectory=packages/fairchem-core

# # 4. Install torch_geometric
# !pip install torch_geometric

In [1]:
import torch, torch_geometric, fairchem.core
from torch_cluster import radius_graph
print("Torch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("Fairchem:", fairchem.core.__version__)
print("radius_graph: OK")

Torch: 2.6.0+cu124
PyG: 2.7.0
Fairchem: 2.0.0
radius_graph: OK


In [ ]:
# !pip install git+https://github.com/facebookresearch/fairchem.git@fairchem_core-2.0.0#subdirectory=packages/fairchem-core

In [4]:
# colab
!git clone https://github.com/yasheshak/Chem-277B-Final-Project.git

Cloning into 'Chem-277B-Final-Project'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 127 (delta 59), reused 85 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 2.16 MiB | 14.18 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [5]:
%cd Chem-277B-Final-Project

/content/Chem-277B-Final-Project/Chem-277B-Final-Project


In [6]:
!ls -a

.			    extract.py		      SchNet_for_import.py
..			    .git		      SchNet_GNN_Baseline.ipynb
db_explore.ipynb	    hyperparam_test.ipynb     SchNet_GNN.ipynb
DimeNet_baseline_new.ipynb  hyperparam_test_v2.ipynb  SchNet_Normalize.ipynb
EDA			    Makefile		      simpleGNN
EDA.ipynb		    README.md
environment.yml		    read_multi_ase.py


Goal: Optimizer hyperparameters

Plan
1. Baseline sweep: identify how sensitive the model is when changing target parameter. A range of values for a single parameter is tested while keeping the other params at its default value.
    - Narrow down hyperparameter values
2. Grid search: explores interactions between hyperparameters by examining every possible combination.

Baseline sweep
1. create dict of start, stop, min (user defined)


In [8]:
# --- Path setup for local modules ---
import sys
sys.path.append('/content/Chem-277B-Final-Project')

# --- Local project modules ---
from read_multi_ase import *
from extract import *
from SchNet_for_import import *

# --- Standard library ---
import glob
from typing import Union

# --- Third-party ---
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import random_split

# --- PyTorch Geometric ---
from torch_geometric.data import Data
from torch_geometric.nn import SchNet
from torch_geometric.loader import DataLoader

# --- Fairchem ---
from fairchem.core.datasets import AseDBDataset

In [9]:
from google.colab import auth
auth.authenticate_user()

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ---------------------------------- Initialize hyperparam range -------------------------------
# parameters to optimize
hidden_channels = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_filters
num_filters = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_interactions
num_interactions = {
    'start': 1,
    'end': 1,
    'step': 1
}
# num_gaussians
num_gaussians = {
    'start': 1,
    'end': 1,
    'step': 1
}
# cutoff
cutoff = {
    'start': 1,
    'end': 1,
    'step': 1
}
# max_num_neighbors
max_num_neighbors = {
    'start': 1,
    'end': 1,
    'step': 1
}

# initialize default params
default_params = {
'hidden_channels': 128,
'num_filters': 128,
'num_interactions': 6,
'num_gaussians': 50,
'cutoff': 5,
'max_num_neighbors': 32
}


params_to_optimize = [hidden_channels, num_filters, num_interactions, num_gaussians, cutoff, max_num_neighbors]

# set parameters
# set_params = [readout, dipole, mean, std, atomref, train_mean, train_std]
default_readout = 'add'
default_dipole = False
default_mean = None
default_std = None
default_atomref = None
default_train_mean = None
default_train_std = None

In [11]:
# ------------------------------ Initialize training set --------------------------------
# given the local dataset path, loads the first .aselmdb file
dataset_path = '/content/drive/MyDrive/train_4M/data0000.aselmdb'
# dataset = AseDBDataset({"src": dataset_path})
files_list = dataset_path
dataset = process_file(files_list, molecule_type = 'biomolecules', max_molecules = 200)

# convert to torch
torch_data = get_data(dataset)
train_dataset, val_dataset = split_data(torch_data, 0.8)

# load
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

Processed 200 atoms


In [12]:
# ------------------------------- Final Function -------------------------------------------
def param_tuning(param: str, param_start: int, param_end: int, param_step: int):
    # initialize numpy array for param vals
    param_vals = np.arange(start=param_start, stop=(param_end+1), step=param_step)

    # initialize params
    params = default_params.copy()

    for val in param_vals:
      # update param dict
      params[param] = val

      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

      #Initialize model with desired parameters
      model = SchNetModel(
          params['hidden_channels'],
          params['num_filters'],
          params['num_interactions'],
          params['num_gaussians'],
          params['cutoff'],
          params['max_num_neighbors'],
          readout = "add",
          dipole = False,
          mean = None,
          std = None,
          atomref = None,
          train_mean = None,
          train_std = None
      ).to(device)

      optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
      loss_function = torch.nn.SmoothL1Loss()

      # run model
      epochs = 50
      train_losses = np.zeros(epochs)
      val_losses = np.zeros(epochs)

      for epoch in range(50):
          train_loss = train(model, train_loader, device, optimizer, loss_function)
          val_loss = evaluate(model, val_loader, device, optimizer, loss_function)

          train_losses[epoch] = train_loss
          val_losses[epoch] = val_loss

      print(f'hyperparameter: {param}, value: {val}')

      # save photo of RMSE graph
      # plot_losses(bio_train_losses, bio_val_losses)
      # save graph

      # save combos to df then excel (maybe graph?)

In [14]:
# ================================== Run Baseline Sweep =================================
#Determine the device to be used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Initialize model with desired parameters
model = SchNetModel().to(device)
#Create ADAM optimizer based on model's parameters and desired learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
#Select loss function for model
loss_function = torch.nn.SmoothL1Loss()

param_tuning('num_interactions', 1, 2, 1)

hyperparameter: num_interactions, value: 1
hyperparameter: num_interactions, value: 2
